<a href="https://colab.research.google.com/github/Dhanshreeshende/Generative_AI/blob/main/practical08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U transformers accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---------- 1. Model and tokenizer initialization ----------
if torch.cuda.is_available():
    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
    DTYPE = torch.float16
else:
    MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"   # lighter model for CPU
    DTYPE = torch.float32
    print("WARNING: No GPU found. Using the smaller 0.5B model on CPU (slow).")

print(f"Loading {MODEL_NAME} ... (first run downloads the model, please wait)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    device_map="auto",
)
model.eval()
print("Model loaded on:", model.device, "\n")

# ---------- 2. Role-based prompt engineering ----------
# SYSTEM prompt = the "role" the model must play.
ROLES = {
    "email": (
        "You are a professional business communication assistant. "
        "You write complete, polite, well-structured emails ready to send. "
        "You never describe or summarize the request; you write the actual email."
    ),
    "report": (
        "You are a professional business analyst and report writer. "
        "You write complete, well-organized reports with clear sections. "
        "You never describe or summarize the request; you write the actual report."
    ),
    "technical": (
        "You are a friendly technical teacher who explains complex topics "
        "to beginners in simple language, using everyday analogies and short examples."
    ),
    "general": (
        "You are a helpful, accurate and clear assistant. "
        "Give complete, relevant answers to the user's request."
    ),
}

# USER prompt templates = task instructions + required structure.
TASKS = {
    "email": (
        "Write a complete professional email for this request:\n"
        "\"{request}\"\n\n"
        "The email MUST contain, in this order:\n"
        "1. Subject: (a clear subject line)\n"
        "2. Greeting (e.g., Dear Sir/Madam or Dear [Name],)\n"
        "3. Body: 2-3 full paragraphs covering the details, reason and any needed action\n"
        "4. Closing (e.g., Yours sincerely, followed by [Your Name])\n\n"
        "Write the email itself now. Do not explain what you are doing."
    ),
    "report": (
        "Write a complete structured report for this request:\n"
        "\"{request}\"\n\n"
        "The report MUST contain these sections with these headings:\n"
        "Title\n"
        "1. Introduction\n"
        "2. Key Findings (use bullet points with realistic example details)\n"
        "3. Recommendations (use bullet points)\n"
        "4. Conclusion\n\n"
        "Write the report itself now. Do not explain what you are doing."
    ),
    "technical": (
        "Explain this topic to a beginner:\n"
        "\"{request}\"\n\n"
        "Your answer MUST contain these three parts with these headings:\n"
        "1. Simple Explanation (3-5 easy sentences)\n"
        "2. Analogy (a real-life comparison)\n"
        "3. Example (a short, concrete example; include a tiny code or "
        "command snippet only if relevant)\n"
    ),
    "general": (
        "{request}\n\n"
        "Give a clear and complete answer."
    ),
}

VALID_MODES = list(ROLES.keys())

# ---------- 3. Content-generation function ----------
MAX_NEW_TOKENS = 600      # upper limit per chunk (NOT a forced minimum)
MAX_CONTINUATIONS = 2     # auto-continue if the answer got cut off

def _eos_ids():
    eos = model.generation_config.eos_token_id
    if eos is None:
        eos = tokenizer.eos_token_id
    return set(eos) if isinstance(eos, (list, tuple)) else {eos}

def generate_content(mode, request):
    """Generate content for the given mode and user request."""
    messages = [
        {"role": "system", "content": ROLES[mode]},
        {"role": "user", "content": TASKS[mode].format(request=request)},
    ]

    # Convert messages into the model's official chat format.
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    full_answer = ""
    eos_set = _eos_ids()

    for _ in range(1 + MAX_CONTINUATIONS):
        inputs = tokenizer(
            prompt_text + full_answer, return_tensors="pt", add_special_tokens=False
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.05,
                pad_token_id=tokenizer.eos_token_id,
            )

        new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
        finished = any(t.item() in eos_set for t in new_ids)
        full_answer += tokenizer.decode(new_ids, skip_special_tokens=True)

        if finished:           # model ended naturally -> done
            break

    return full_answer.strip()

# ---------- 4. Interactive chatbot loop ----------
def chatbot():
    print("=" * 60)
    print("  AI CONTENT GENERATION CHATBOT")
    print("  Modes: email | report | technical | general")
    print("  Type 'quit' at any prompt to exit.")
    print("=" * 60)

    while True:
        try:
            mode = input("\nSelect mode (email/report/technical/general): ").strip().lower()
            if mode == "quit":
                break
            if mode not in VALID_MODES:
                print(f"Invalid mode '{mode}'. Please choose one of: {', '.join(VALID_MODES)}")
                continue

            request = input("Enter your topic or request: ").strip()
            if request.lower() == "quit":
                break
            if not request:
                print("Request cannot be empty. Please try again.")
                continue

            print("\nGenerating... please wait.\n")
            result = generate_content(mode, request)

            if not result:
                print("The model returned an empty response. Try rephrasing your request.")
                continue

            print("-" * 60)
            print(f"[{mode.upper()} OUTPUT]")
            print("-" * 60)
            print(result)
            print("-" * 60)

        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print("GPU out of memory. Try a shorter request, or restart the runtime "
                  "and use the 0.5B model (see troubleshooting).")
        except KeyboardInterrupt:
            print("\nInterrupted by user.")
            break
        except Exception as e:
            print(f"Unexpected error: {type(e).__name__}: {e}")

    print("\nGoodbye!")

chatbot()

Loading Qwen/Qwen2.5-1.5B-Instruct ... (first run downloads the model, please wait)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on: cuda:0 

  AI CONTENT GENERATION CHATBOT
  Modes: email | report | technical | general
  Type 'quit' at any prompt to exit.

Select mode (email/report/technical/general): email
Enter your topic or request: Write an email to my manager requesting 2 days of sick leave due to fever

Generating... please wait.

------------------------------------------------------------
[EMAIL OUTPUT]
------------------------------------------------------------
Subject: Request for 2 Days of Sick Leave Due to Fever

Dear Manager,

I am writing to kindly request your approval for 2 days of sick leave due to the onset of a fever. I have been feeling unwell with high temperatures and muscle aches, which I believe are consistent with the flu. I have consulted with my healthcare provider who recommended that I rest and stay home until I feel better.

I understand that my absence will cause inconvenience to the team and our workload, but please allow me to take this time to care for myself prop